# Pedestrian Behaviour — QR-DQN
Trains the pedestrian counter-agent () used by the crossing environments.


In [5]:
# Imports
import numpy as np
import cv2
import matplotlib.pyplot as plt
import PIL.Image as Image
import gymnasium as gym
import random
from gymnasium import Env, spaces
import time

font = cv2.FONT_HERSHEY_COMPLEX_SMALL

# src modules
from src.motion_models import next_position_sim, Particle_ped, foward_sim
from src.environments import Pedestrian_Behaviour, set_ped_model

In [ ]:
# Sanity check — random policy on the pedestrian env
env = gym.make("PedestrianBehaviour-v1")
obs, _ = env.reset()   # gymnasium reset() returns (obs, info)
print(obs)
total_reward = 0
done = False
while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    if not done:
        total_reward += reward
    print(f"{obs} {env.unwrapped.conflict} {action}  {reward} {done}")
print(total_reward)

NameNotFound: Environment `Pedestrian_Behaviour` doesn't exist. Did you mean: `PedestrianBehaviour`?

In [ ]:
# SB3 imports
#!pip install stable-baselines3[extra]
#!pip install sb3-contrib
from stable_baselines3 import DQN
from sb3_contrib import QRDQN

In [ ]:
# Train pedestrian QR-DQN model
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnRewardThreshold
import warnings
warnings.filterwarnings("ignore")
callback_on_best = StopTrainingOnRewardThreshold(reward_threshold=18, verbose=1)
env = gym.make("Pedestrian_Behaviour-v1")
eval_callback = EvalCallback(env, eval_freq=1000,
                             deterministic=False, callback_on_new_best=callback_on_best)
model = QRDQN("MlpPolicy", env, verbose=0, learning_rate=0.0001, gamma=0.99)
model.learn(total_timesteps=80000, log_interval=3, progress_bar=True, callback=eval_callback)

In [ ]:
# Save trained model
model.save("models/ped_model")


In [ ]:
# Load model
model = QRDQN.load("models/ped_model")

In [ ]:
# Evaluate model
obs, _ = env.reset()
print(obs)
done = False
while not done:
    action = model.predict(obs, deterministic=False)
    obs, reward, terminated, truncated, _ = env.step(action[0])
    done = terminated or truncated
    print(f"state:{obs} - action: {action[0]} - {env.unwrapped.conflict} - reward:{reward}  {done}")

In [ ]:
# Single-step inference check
ob = [24.377193, 55.25732, 44.06437, 5, 1, 0.5992604]
print(model.predict(ob, deterministic=False))

In [ ]:
# Visualise trajectory
import matplotlib.pyplot as plt

def plot_ped_traj(car_positions_x, car_positions_y,
                  ped_positions_x, ped_positions_y, y=True):
    plt.figure(figsize=(10, 2))
    plt.plot(car_positions_x, car_positions_y, label='Car Trajectory', marker='o', color='blue')
    yielding = "yielding"
    plt.plot(ped_positions_x, ped_positions_y, label='Pedestrian Trajectory', marker='x', color='red')
    if not y:
        yielding = "constant speed"
    plt.title(f'Pedestrian and Car Movement in Coordinate Space {yielding} situation')
    plt.xlabel('X Position')
    plt.ylabel('Y Position')
    plt.ylim(-4, 8)
    plt.legend()
    plt.grid(True)
    plt.show()

obs, _ = env.reset()

car_positions_x, car_positions_y = [], []
ped_positions_x, ped_positions_y = [], []
time_steps = []
car_speed, ped_speed = [], []
print("Initial Observation:", obs)
done = False
t = 0
while not done:
    action = model.predict(obs, deterministic=False)
    obs, reward, terminated, truncated, _ = env.step(action[0])
    done = terminated or truncated

    car_x = obs[0]
    car_speed.append(obs[1])
    ped_speed.append(obs[4])
    ped_x = obs[2]
    ped_y = obs[3]

    car_positions_x.append(car_x)
    car_positions_y.append(0)
    ped_positions_x.append(ped_x)
    ped_positions_y.append(ped_y)
    time_steps.append(t)

    print(f"Time Step: {t}, State: {obs} - Action: {action[0]} - reward:{reward} - done:{done}")
    t += 1

plot_ped_traj(car_positions_x, car_positions_y, ped_positions_x, ped_positions_y,
              y=env.unwrapped.conflict)

In [ ]:
# Speed plot
import matplotlib.pyplot as plt
import numpy as np

time = range(t)
print(ped_speed)
pedestrian_speed = ped_speed

fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(time, pedestrian_speed, label='Pedestrian Speed', color='red', linewidth=2, marker='x')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Speed (m/s)')
ax1.grid(True)
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.plot(time, car_speed, label='Car Speed', color='blue', linewidth=2, marker='o')
ax2.set_ylabel('Car Speed (m/s)', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

plt.title('Pedestrian and Car Speed over Time')
fig.tight_layout()
plt.show()